# 14 Frozen Encoder Eval

**Purpose.** Evaluate cached embeddings with ridge / XGBoost heads under the standard protocol.

**Thesis reference.** §5 Representation, Not Model

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
# severity_all
import json, time, os
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as tvm

# config
IMAGE_ROOT      = "images224"
LABELS          = "labels/labels_severity_all.json"
OUT_DIR         = "embeddings_severity"
REPRESENTATIONS = ["grayscale_heatmap", "rgb_mz_split"]
MODELS          = ["resnet18", "efficientnet_b0", "vit_b_16"]
BATCH_SIZE      = 8
ID_AUC_RANK     = 0.535     # patient identifier, rank statistic
ID_AUC_BLOCK    = 0.582     # patient identifier, ten blocks
CLASSICAL       = 0.741     # best classical model on this task

REG = {"resnet18":        (tvm.resnet18,        "ResNet18_Weights",        "fc"),
       "resnet50":        (tvm.resnet50,        "ResNet50_Weights",        "fc"),
       "efficientnet_b0": (tvm.efficientnet_b0, "EfficientNet_B0_Weights", "classifier"),
       "vit_b_16":        (tvm.vit_b_16,        "ViT_B_16_Weights",        "heads")}

def build(name):
    ctor, wname, head = REG[name]
    w = getattr(tvm, wname).DEFAULT
    m = ctor(weights=w)
    if head == "classifier" and isinstance(getattr(m, head), nn.Sequential):
        seq = list(getattr(m, head))
        while seq and not isinstance(seq[-1], nn.Linear): seq.pop()
        if seq: seq.pop()
        setattr(m, head, nn.Sequential(*seq) if seq else nn.Identity())
    else:
        setattr(m, head, nn.Identity())
    m.eval()
    for p in m.parameters(): p.requires_grad_(False)
    try: size = w.transforms().crop_size[0]
    except Exception: size = 224
    return m, size

class DS(Dataset):
    def __init__(self, d, pids, size):
        self.d, self.pids = Path(d), pids
        self.tf = transforms.Compose([transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        return self.tf(Image.open(self.d/f"{self.pids[i]}.png").convert("RGB"))

lab = json.load(open(LABELS))["labels"]
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"torch {torch.__version__}, threads {torch.get_num_threads()}")
print(f"task: {Path(LABELS).stem}  ({len(lab)} labelled)\n")

t_start = time.time()
for rep in REPRESENTATIONS:
    d = Path(IMAGE_ROOT)/rep
    if not d.is_dir(): print(f"[skip] {d} missing"); continue
    pids = [p for p in sorted(lab) if (d/f"{p}.png").exists()]
    print(f"{rep}: {len(pids)} patients")
    for mn in MODELS:
        dest = Path(OUT_DIR)/f"{rep}__{mn}.npy"
        if dest.exists(): print(f"  {mn}: cached"); continue
        try:
            model, size = build(mn)
            dl = DataLoader(DS(d, pids, size), batch_size=BATCH_SIZE, shuffle=False)
            out, t0, done = [], time.time(), 0
            with torch.no_grad():
                for x in dl:
                    z = model(x)
                    out.append((z.flatten(1) if z.ndim > 2 else z).cpu().numpy())
                    done += len(x); print(f"    {done}/{len(pids)}", end="\r", flush=True)
            emb = np.concatenate(out, 0)
            np.save(dest, emb)
            json.dump({"representation": rep, "model": mn, "pids": pids,
                       "dim": int(emb.shape[1])}, open(dest.with_suffix(".json"), "w"))
            print(f"  {mn}: {emb.shape} in {(time.time()-t0)/60:.1f} min")
        except Exception as e:
            print(f"  {mn}: FAILED {type(e).__name__}: {e}")
    print()
print(f"extraction took {(time.time()-t_start)/60:.0f} min\n")

# evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

mk = lambda: Pipeline([("s", StandardScaler()),
                       ("c", LogisticRegression(C=0.1, max_iter=3000,
                                                class_weight="balanced"))])
print(f"{'representation':20s} {'model':17s} {'dim':>5} {'AUC':>7} "
      f"{'95% CI':>16} {'vs block':>9} {'vs classical':>13}")
print("-"*95)
rows = []
for f in sorted(Path(OUT_DIR).glob("*.npy")):
    meta = f.with_suffix(".json")
    if not meta.exists(): continue
    info = json.load(open(meta))
    X = np.load(f); y = np.array([lab[p] for p in info["pids"]])
    if len(np.unique(y)) < 2: continue
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                                          random_state=42).split(X, y):
        oof[te] += mk().fit(X[tr], y[tr]).predict_proba(X[te])[:,1]; cnt[te]+=1
    o = oof/cnt; auc = roc_auc_score(y, o)
    rng = np.random.default_rng(0); bs = []
    for _ in range(2000):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) > 1: bs.append(roc_auc_score(y[i], o[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{info['representation']:20s} {info['model']:17s} {X.shape[1]:5d} "
          f"{auc:7.4f} [{lo:.3f},{hi:.3f}] {auc-ID_AUC_BLOCK:+9.4f} "
          f"{auc-CLASSICAL:+13.4f}")
    rows.append((info['representation'], info['model'], auc, lo, hi))

if rows:
    b = max(rows, key=lambda r: r[2])
    print(f"\nbest image result : {b[0]} / {b[1]} = {b[2]:.4f}  [{b[3]:.3f}, {b[4]:.3f}]")
    print(f"best classical    : {CLASSICAL:.4f}  (coarse 2-D map)")
    print(f"identifier, rank  : {ID_AUC_RANK:.4f}")
    print(f"identifier, block : {ID_AUC_BLOCK:.4f}   <- the stricter baseline")
    ret = (b[2]-0.5)/(CLASSICAL-0.5)
    print(f"\nsignal retained relative to the best classical model: {ret:.0%}")
    if b[3] <= ID_AUC_BLOCK:
        print("-> the interval includes the block baseline; not a result")
    elif b[2] > CLASSICAL:
        print("-> exceeds the classical model: the finer retention-time axis")
        print("   outweighs the mass resolution given up")
    else:
        print("-> clears the baseline but does not beat the coarse map")

torch 2.8.0+cu128, threads 32
task: labels_severity_all  (771 labelled)

grayscale_heatmap: 510 patients
  resnet18: (510, 512) in 0.1 min
  efficientnet_b0: (510, 1280) in 0.1 min
  vit_b_16: (510, 768) in 0.2 min

rgb_mz_split: 510 patients
  resnet18: (510, 512) in 0.1 min
  efficientnet_b0: (510, 1280) in 0.1 min
  vit_b_16: (510, 768) in 0.2 min

extraction took 1 min

representation       model               dim     AUC           95% CI  vs block  vs classical
-----------------------------------------------------------------------------------------------
grayscale_heatmap    efficientnet_b0    1280  0.6340 [0.574,0.692]   +0.0520       -0.1070
grayscale_heatmap    resnet18            512  0.5368 [0.481,0.596]   -0.0452       -0.2042
grayscale_heatmap    vit_b_16            768  0.5780 [0.519,0.636]   -0.0040       -0.1630
rgb_mz_split         efficientnet_b0    1280  0.6457 [0.589,0.701]   +0.0637       -0.0953
rgb_mz_split         resnet18            512  0.6103 [0.554,0.667]   

In [2]:
# nested cross-validation
import json, time, warnings
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

LABELS = "labels/labels_severity_all.json"
PROF   = "profiles"
CURRENT_BEST = 0.741        # fixed-configuration result now in the thesis
ID_BLOCK     = 0.582

lab  = json.load(open(LABELS))["labels"]
pids = json.load(open(f"{PROF}/profile_pids.json"))
idx  = [i for i,p in enumerate(pids) if p in lab]
y    = np.array([lab[pids[i]] for i in idx])
print(f"n = {len(y)}  ({y.sum()} mCKD / {(1-y).sum()} eCKD)\n")

F = np.load(f"{PROF}/mz_profiles.npy")[idx].astype(np.float64)
C = np.load(f"{PROF}/coarse_maps.npy")[idx].astype(np.float64)

def norm(M):
    t = M.sum(1, keepdims=True); t[t==0]=1
    return np.log1p(M/t*1e6)

REPS = {
  "profile_1d":  norm(F),
  "map_2d":      norm(C.reshape(len(C),-1)),
  "rt_profile":  norm(C.sum(axis=2)),
  "mz_from_map": norm(C.sum(axis=1)),
}
try:
    E = {f.stem: np.load(f) for f in Path("embeddings_severity").glob("*.npy")}
    for k,v in E.items():
        meta = json.load(open(Path("embeddings_severity")/f"{k}.json"))
        order = {p:i for i,p in enumerate(meta["pids"])}
        sel = [order[pids[i]] for i in idx if pids[i] in order]
        if len(sel) == len(y): REPS[f"emb_{k}"] = v[sel]
except Exception as e:
    print(f"(embeddings not loaded: {e})")
print("representations:", ", ".join(REPS), "\n")

def select(Xtr, Xte, k):
    if Xtr.shape[1] <= k: return Xtr, Xte
    o = np.argsort(Xtr.var(0))[::-1][:k]
    return Xtr[:,o], Xte[:,o]

def ridge(c):  return Pipeline([("s",StandardScaler()),
    ("c",LogisticRegression(C=c,max_iter=4000,class_weight="balanced"))])
class PLS:
    def __init__(s,n): s.n=n
    def fit(s,X,yy):
        s.m=PLSRegression(n_components=min(s.n,X.shape[1]-1)).fit(X,yy); return s
    def predict_proba(s,X):
        p=s.m.predict(X).ravel(); return np.c_[1-p,p]
def pls(n): return Pipeline([("s",StandardScaler()),("c",PLS(n))])

GRID = ([("ridge",{"C":c},lambda c=c: ridge(c)) for c in (0.003,0.01,0.03,0.1,0.3,1.0)] +
        [("plsda",{"nc":n},lambda n=n: pls(n))  for n in (2,3,5,8,12)])
try:
    import xgboost as xgb
    def xg(d,e,n):
        return xgb.XGBClassifier(n_estimators=n,max_depth=d,learning_rate=e,
            subsample=0.8,colsample_bytree=0.5,eval_metric="logloss",
            verbosity=0,n_jobs=4)
    GRID += [("xgb",{"depth":d,"lr":e},lambda d=d,e=e: xg(d,e,300))
             for d in (2,3) for e in (0.03,0.1)]
    print(f"{len(GRID)} configurations (xgboost available)")
except ImportError:
    print(f"{len(GRID)} configurations (no xgboost)")

NFEAT = [1000, 3000, 10000]
OUTER = StratifiedKFold(5, shuffle=True, random_state=0)

# flat evaluation of every configuration
print(f"\n{'representation':22s} {'model':8s} {'params':22s} {'k':>6s} {'AUC':>7}")
print("-"*74)
t0 = time.time(); flat = []
for rn, X in REPS.items():
    for k in ([f for f in NFEAT if f < X.shape[1]] or [X.shape[1]]):
        for mn, par, mk in GRID:
            oof = np.zeros(len(y))
            for tr,te in OUTER.split(X,y):
                A,B = select(X[tr], X[te], k)
                oof[te] = mk().fit(A,y[tr]).predict_proba(B)[:,1]
            a = roc_auc_score(y,oof)
            flat.append((rn,mn,str(par),k,a))
            print(f"{rn:22s} {mn:8s} {str(par):22s} {k:6d} {a:7.4f}")
print(f"\n{len(flat)} configurations in {(time.time()-t0)/60:.1f} min")

flat.sort(key=lambda r:-r[4])
print(f"\nbest single configuration: {flat[0][4]:.4f}  "
      f"({flat[0][0]}, {flat[0][1]}, {flat[0][2]}, k={flat[0][3]})")
print(f"thesis figure (fixed config): {CURRENT_BEST:.4f}")

# expected maximum of this many noisy estimates, under the null
n1,n2 = int(y.sum()), int((1-y).sum())
se = np.sqrt((n1+n2+1)/(12*n1*n2))
m = len(flat)
print(f"\nexpected maximum over {m} configurations under the null: "
      f"{0.5+se*np.sqrt(2*np.log(2*m)):.4f}")
print("  the best flat result must clearly exceed this to mean anything")

# nested: selection inside the outer loop
print("\nNESTED ESTIMATE (configuration chosen inside each outer fold)")
oof = np.zeros(len(y)); chosen = []
for f,(tr,te) in enumerate(OUTER.split(np.zeros(len(y)),y),1):
    best, bs = None, -1
    inner = StratifiedKFold(4, shuffle=True, random_state=1)
    for rn,X in REPS.items():
        for k in ([q for q in NFEAT if q < X.shape[1]] or [X.shape[1]]):
            for mn,par,mk in GRID:
                p = np.zeros(len(tr))
                for itr,ite in inner.split(X[tr],y[tr]):
                    A,B = select(X[tr][itr], X[tr][ite], k)
                    p[ite] = mk().fit(A,y[tr][itr]).predict_proba(B)[:,1]
                sc = roc_auc_score(y[tr],p)
                if sc > bs: bs, best = sc, (rn,mn,par,k,mk)
    rn,mn,par,k,mk = best
    A,B = select(REPS[rn][tr], REPS[rn][te], k)
    oof[te] = mk().fit(A,y[tr]).predict_proba(B)[:,1]
    chosen.append((rn,mn,str(par),k))
    print(f"  fold {f}: chose {rn}/{mn}/{par} k={k}  (inner {bs:.4f})")

nested = roc_auc_score(y,oof)
rng=np.random.default_rng(0); bs=[]
for _ in range(2000):
    i=rng.choice(len(y),len(y),replace=True)
    if len(np.unique(y[i]))>1: bs.append(roc_auc_score(y[i],oof[i]))
lo,hi=np.percentile(bs,[2.5,97.5])

print(f"\n  nested AUC        {nested:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  thesis figure     {CURRENT_BEST:.4f}   (fixed configuration)")
print(f"  block baseline    {ID_BLOCK:.4f}")
print(f"  best flat search  {flat[0][4]:.4f}   (optimistic)")
print(f"\n  folds agreed on the representation: "
      f"{len(set(c[0] for c in chosen))} distinct chosen")
if lo > ID_BLOCK:
    print("  -> clears the block baseline")
if nested > CURRENT_BEST + 0.02:
    print("  -> materially better than the fixed configuration; worth adopting")
elif nested < CURRENT_BEST - 0.02:
    print("  -> lower than the fixed configuration, which is the usual direction;")
    print("     the nested figure is the honest one")
else:
    print("  -> indistinguishable from the fixed configuration: the simple")
    print("     model was already at the ceiling these data support")

n = 510  (121 mCKD / 389 eCKD)

representations: profile_1d, map_2d, rt_profile, mz_from_map, emb_grayscale_heatmap__efficientnet_b0, emb_grayscale_heatmap__resnet18, emb_rgb_mz_split__efficientnet_b0, emb_grayscale_heatmap__vit_b_16, emb_rgb_mz_split__vit_b_16, emb_rgb_mz_split__resnet18 

15 configurations (xgboost available)

representation         model    params                      k     AUC
--------------------------------------------------------------------------
profile_1d             ridge    {'C': 0.003}             1000  0.5564
profile_1d             ridge    {'C': 0.01}              1000  0.5528
profile_1d             ridge    {'C': 0.03}              1000  0.5481
profile_1d             ridge    {'C': 0.1}               1000  0.5447
profile_1d             ridge    {'C': 0.3}               1000  0.5433
profile_1d             ridge    {'C': 1.0}               1000  0.5401


AttributeError: The following error was raised: 'PLS' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.